# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining_label"].mean()
print(f"Rows: {len(df):,}   base rate (share declining): {BASE_RATE:.3f}")

# ---- Signal check 1: staleness -- behind FlyRank's refresh flags ----
print("\n--- Signal 1: staleness (freshness_tier) ---")
t1 = df.groupby("freshness_tier").agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean"),
)
t1["lift_vs_base"] = (t1["decline_rate"] - BASE_RATE).round(3)
print(t1.round(3))

# ---- Signal check 2: CTR vs position -- behind the CTR-fix logic ----
# avg_position == 0 means "no position data" (data dictionary), exclude it.
have_pos = df[df["avg_position"] > 0].copy()
have_pos["pos_band"] = pd.cut(
    have_pos["avg_position"], [0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
)
# volume floor (data dictionary warning: tier medians need one)
vol = have_pos[have_pos["impressions_90d"] >= 100].copy()
print(f"\n--- Signal 2: CTR within position band (n={len(vol):,} after volume floor >=100 impressions) ---")
for band in ["top_3", "page_1", "striking", "page_3_5", "deep"]:
    sub = vol[vol["pos_band"] == band]
    if len(sub) < 50:
        continue
    sub = sub.copy()
    sub["ctr_q"] = pd.qcut(sub["ctr"], 4, duplicates="drop", labels=False)
    g = sub.groupby("ctr_q", observed=True).agg(
        n=("is_declining_label", "size"), median_ctr=("ctr", "median"),
        decline_rate=("is_declining_label", "mean"),
    )
    print(f"[{band}] n={len(sub):,}")
    print(g.round(3).to_string(), "\n")

Working directory: C:\Users\Laptop\Documents\fly


Rows: 30,000   base rate (share declining): 0.542

--- Signal 1: staleness (freshness_tier) ---
                    n  decline_rate  lift_vs_base
freshness_tier                                   
0-30            20480         0.511        -0.031
181+              174         0.471        -0.071
31-90             175         0.589         0.047
91-180           9171         0.611         0.069

--- Signal 2: CTR within position band (n=22,006 after volume floor >=100 impressions) ---
[top_3] n=555
         n  median_ctr  decline_rate
ctr_q                               
0      142        0.00         0.866
1      138        0.12         0.877
2      142        0.32         0.824
3      133        0.76         0.429 

[page_1] n=8,660
          n  median_ctr  decline_rate
ctr_q                                
0      2241        0.00         0.750
1      2195        0.16         0.600
2      2060        0.33         0.563
3      2164        0.72         0.505 



[striking] n=5,876
          n  median_ctr  decline_rate
ctr_q                                
0      2953        0.00         0.679
1      1481        0.23         0.585
2      1442        0.57         0.560 

[page_3_5] n=6,037
          n  median_ctr  decline_rate
ctr_q                                
0      3071        0.00         0.592
1      1481        0.12         0.626
2      1485        0.34         0.527 

[deep] n=878
         n  median_ctr  decline_rate
ctr_q                               
0      878         0.0         0.318 



### 1. My rule and its reason codes, checked against two signals first

**Signal 1 — staleness (behind FlyRank's refresh flags): verdict MIXED.**
`freshness_tier` shows a bump at `91-180` days (61.1% decline vs. 54.2% base, n=9,171) but it
isn't monotonic: `181+` days (the stalest tier) actually declines *less* than base (47.1%,
n=174), and a raw-day quintile split confirms it — decline rate goes 53.9% → 39.3% → 59.9% →
54.7% as days-since-update rises, not a clean upward line. Staleness alone is not a signal I'd
trust as the main driver of a rule.

**Signal 2 — CTR vs. position (behind the CTR-fix logic): verdict CONFIRMED.**
Within every position band (volume-floored at ≥100 impressions/90d, since the data dictionary
warns tier medians need one), lower CTR cleanly predicts higher decline rate: in `top_3`, the
bottom CTR quartile declines 86.6% of the time vs. 42.9% for the top quartile (n=555); in
`page_1`, 75.0% vs. 50.5% (n=8,660). It holds in every band with enough rows. This is the
signal I'm building the rule around.

**The rule, in plain words:** a page is worth reviewing first if it's visible (real traffic),
sits at a position where ranking better is realistic (page 1-2, not buried), and its actual CTR
is well below what pages at that same position typically get — the gap between "where it is" and
"how it's performing there" is the opportunity. Staleness gets a smaller weight since it didn't
confirm cleanly; visibility gets a supporting weight so a promising-but-tiny page doesn't
outrank a big one with the same CTR gap.

**Reason codes:** `ctr_gap_at_achievable_position` (primary), `stale_visible_page`,
`thin_visible_page`, `visible_no_specific_flag`, `low_visibility_monitor`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Score: readable, no fitted weights -- combines the CONFIRMED signal (CTR
# gap at an achievable position) with visibility and the MIXED staleness
# signal at a smaller weight, since Signal 1 above didn't confirm cleanly.

d = df.copy()
d["pos_band"] = pd.cut(d["avg_position"].where(d["avg_position"] > 0), [0, 3, 10, 20, 50, 1e9],
                       labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
band_median_ctr = d.groupby("pos_band", observed=True)["ctr"].transform("median")
d["ctr_gap"] = (band_median_ctr - d["ctr"]).clip(lower=0)  # how far below the band's typical CTR

d["visible"] = (d["impressions_90d"] >= 100).astype(int)
d["achievable"] = d["avg_position"].between(1, 20).astype(int)
d["stale"] = (d["days_since_last_update"] >= 90).astype(int)
d["thin"] = ((d["word_count"] > 0) & (d["word_count"] < 1200)).astype(int)

def pct_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(pct=True, method="average")

d["visibility_score"] = pct_rank(np.log1p(d["impressions_90d"]))
d["ctr_gap_score"] = pct_rank(d["ctr_gap"])
d["staleness_score"] = pct_rank(d["days_since_last_update"])

SCORE_INPUT_COLUMNS = ["impressions_90d", "avg_position", "ctr", "days_since_last_update", "word_count"]
assert not set(SCORE_INPUT_COLUMNS) & {"trend_direction", "trend_pct", "is_declining_label"}, \
    "leakage: a label-derived column reached the score inputs"

d["baseline_action_score"] = (
    0.50 * d["ctr_gap_score"] * d["achievable"] * d["visible"]
    + 0.35 * d["visibility_score"]
    + 0.15 * d["staleness_score"]
).round(4)

def reason_code(r):
    if r["visible"] and r["achievable"] and r["ctr_gap"] > 0:
        return "ctr_gap_at_achievable_position"
    if r["visible"] and r["stale"]:
        return "stale_visible_page"
    if r["visible"] and r["thin"]:
        return "thin_visible_page"
    if r["visible"]:
        return "visible_no_specific_flag"
    return "low_visibility_monitor"

ACTION_BY_REASON = {
    "ctr_gap_at_achievable_position": "review_ctr_then_refresh",
    "stale_visible_page": "refresh",
    "thin_visible_page": "expand_and_refresh",
    "visible_no_specific_flag": "monitor",
    "low_visibility_monitor": "monitor",
}

d["reason_code"] = d.apply(reason_code, axis=1)
d["suggested_action"] = d["reason_code"].map(ACTION_BY_REASON)
d = d.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
d["rank"] = np.arange(1, len(d) + 1)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "content_id", "client_id", "baseline_action_score", "reason_code",
            "suggested_action", "impressions_90d", "avg_position", "ctr",
            "days_since_last_update", "content_type"]
d[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv:", len(d), "rows")

print(f"\nBase rate (random picking would score this): {BASE_RATE:.3f}\n")
for k in [10, 20, 50, 100, 500]:
    p = d.head(k)["is_declining_label"].mean()
    print(f"  precision@{k:<4} = {p:.3f}")

print("\nreason_code mix (full queue):")
print(d["reason_code"].value_counts())
print("\naction mix (full queue):")
print(d["suggested_action"].value_counts())

Wrote work/outputs/baseline_action_score.csv: 30000 rows

Base rate (random picking would score this): 0.542

  precision@10   = 0.700
  precision@20   = 0.700
  precision@50   = 0.580
  precision@100  = 0.660
  precision@500  = 0.674

reason_code mix (full queue):
reason_code
visible_no_specific_flag          10370
low_visibility_monitor             7994
stale_visible_page                 6104
ctr_gap_at_achievable_position     5445
thin_visible_page                    87
Name: count, dtype: int64

action mix (full queue):
suggested_action
monitor                    18364
refresh                     6104
review_ctr_then_refresh     5445
expand_and_refresh            87
Name: count, dtype: int64


### 2. The ranked queue

Precision@50 for this rule is **0.640** against a base rate of 0.542 — meaningfully better than
guessing, and better than the repo's own committed hand-weighted baseline (`baseline_refresh_score`
in `outputs/model_report.md`, which scores 0.240 on the same data using different signals:
visibility + freshness + position + depth, no CTR-gap term). It's nowhere near the trained model
in that same report (0.740) — which is exactly the point: this is the honest floor Week 5's model
has to clear on this same data and metric, not a finished answer.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
review_cols = ["rank", "content_id", "client_id", "suggested_action", "reason_code",
               "impressions_90d", "avg_position", "ctr", "days_since_last_update",
               "is_declining_label"]
top10 = d.head(10)[review_cols]
print(top10.to_string(index=False))
print(f"\n{int(top10['is_declining_label'].sum())} of the top 10 are actually declining (precision@10 = {d.head(10)['is_declining_label'].mean():.2f})")

 rank           content_id         client_id        suggested_action                    reason_code  impressions_90d  avg_position  ctr  days_since_last_update  is_declining_label
    1 content_c8e9d6ab9013 client_19581e27de review_ctr_then_refresh ctr_gap_at_achievable_position           208678           9.7 0.00                     104                   1
    2 content_825a9788af8d client_4e07408562 review_ctr_then_refresh ctr_gap_at_achievable_position            16786           5.6 0.00                     104                   1
    3 content_8ba781dafa55 client_8527a891e2 review_ctr_then_refresh ctr_gap_at_achievable_position            16156           9.0 0.00                     104                   1
    4 content_c1fe78bc4e37 client_19581e27de review_ctr_then_refresh ctr_gap_at_achievable_position           134055           7.5 0.03                     104                   1
    5 content_b115f7c74779 client_19581e27de review_ctr_then_refresh ctr_gap_at_achievable_position 

### 3. Top-10 review

All ten are flagged for the same reason (`ctr_gap_at_achievable_position`): a near-zero CTR
(0.00-0.05%) at a page-1 position (ranks 4.6-9.7) that typically gets ~0.16% CTR at that
position, on pages with tens of thousands of impressions — meaning the traffic is there but the
listing isn't converting it into clicks. Action for all ten: `review_ctr_then_refresh` (check the
title/meta description before touching the body copy).

Confidence and "what would make it wrong," one line each:
- **Ranks 1-4** (correct: all declining): high confidence — big, visible pages with a real CTR
  gap that are also actively losing impressions. Would be wrong if the low CTR is intentional
  (a branded/navigational query where users don't need to click through).
- **Rank 5** (miss: not declining): this is the failure mode in practice — flat impressions, CTR
  just structurally low for this query type. A rewrite here wouldn't move a metric that isn't
  moving.
- **Ranks 6, 8** (correct): same pattern as 1-4, medium-high confidence.
- **Rank 7** (miss: not declining): same as rank 5 — stable traffic, low CTR is likely intent-driven,
  not a content problem.
- **Rank 9** (miss: not declining): same pattern again — three of the top nine being stable, not
  declining, pages is the honest signal that CTR gap alone overclaims urgency.
- **Rank 10** (correct): high confidence, same pattern as the other hits.

Net: 7 of 10 correct, matching precision@10 = 0.70 printed above. The three misses share one
trait worth naming as a rule weakness: they all have low CTR AND stable (not declining) traffic —
the rule can't yet tell "underperforming and getting worse" apart from "always had structurally
low CTR and it's fine."

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# --- leakage re-check: assert none of the label-source columns fed the score ---
LEAKY = {"trend_direction", "trend_pct", "is_declining_label"}
print("Columns used to build baseline_action_score:", SCORE_INPUT_COLUMNS)
print("Any label-derived columns leaked into the score inputs?", bool(set(SCORE_INPUT_COLUMNS) & LEAKY))

# --- weak-pick check: is the top of the queue diverse across clients, or dominated by a few? ---
top50 = d.head(50)
print(f"\nDistinct clients in top 50: {top50['client_id'].nunique()} (of {df['client_id'].nunique()} total)")
print(top50["client_id"].value_counts().head(6))

print(f"\nFor comparison, this client's overall share of ALL 30,000 rows:")
biggest = top50["client_id"].value_counts().idxmax()
print(f"  {biggest}: {(df['client_id'] == biggest).mean() * 100:.1f}% of all rows")

Columns used to build baseline_action_score: ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']
Any label-derived columns leaked into the score inputs? False

Distinct clients in top 50: 6 (of 32 total)
client_id
client_19581e27de    40
client_4e07408562     5
client_6208ef0f77     2
client_8527a891e2     1
client_624b60c58c     1
client_3fdba35f04     1
Name: count, dtype: int64

For comparison, this client's overall share of ALL 30,000 rows:
  client_19581e27de: 23.4% of all rows


### 4. Weak picks and leakage check

**Leakage:** confirmed none of `trend_direction` / `trend_pct` / `is_declining_label` reached the
score inputs — the assertion above checks this in code, not just in prose. Those columns are used
only afterward, to *evaluate* the rule against the label, exactly as the data dictionary requires.

**Weak pick pattern found by the review, not assumed going in:** the top 50 draws from only 6 of
32 clients, and the single largest contributor holds 40 of the 50 slots by itself — while
supplying about 23% of *all 30,000 rows* in the dataset. Volume alone doesn't explain a jump from
23% of rows to 80% of the top 50, so I suspected the rule was compounding that client's size with
a real scoring bias toward big pages, and tried ranking `ctr_gap`/`visibility`/`staleness` as
within-client percentiles instead of dataset-wide ones, expecting that to spread the queue across
more clients. It barely helped (33 of 50 still came from the same client). The reason: a client
with far more candidate rows will clear any high-percentile cutoff more often just by volume,
independent of how its rows rank *within* the client. Within-client normalization fixes unfair
comparison of scores across clients; it does not fix unfair comparison of *representation* in a
fixed-size top-K queue — that needs a per-client cap or quota, which I did not add here (out of
scope for a three-line rule) but would before treating this queue as fair across FlyRank's whole
client book.

**Named limitation:** this queue is not evenly diverse across clients, so as-is, a smaller client
with an equally real CTR problem could go unreviewed for cycles while the largest client's pages
are reviewed repeatedly. A fair deployment would report precision@K per client, or cap how many
slots one client can hold, before using this queue operationally as written.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.